# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading, exploring, and processing the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library.

### Dataset Source
The dataset source is defined by its Croissant schema at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

The Croissant schema organizes tabular data via RecordSets, Fields, and Columns. We'll list their `@id`s and show a preview.

In [ ]:
# Explore schema: List record sets and their fields
record_sets = metadata.record_set
if record_sets is not None and len(record_sets) > 0:
    print(f"Found {len(record_sets)} RecordSets:")
    for rs in record_sets:
        print(f"- RecordSet @id: {rs['@id']}")
        if 'field' in rs:
            print("  Fields:")
            for fld in rs['field']:
                print(f"    - Field @id: {fld['@id']}, name: {fld.get('name', '')}, dataType: {fld.get('dataType', '')}")
else:
    print("No RecordSets found in metadata. Check schema definition.")

# Preview first few records for each record set
if record_sets is not None and len(record_sets) > 0:
    for rs in record_sets:
        print(f"\nPreview records for RecordSet {rs['@id']}:")
        records = dataset.records(record_set=rs['@id'])
        for i, rec in enumerate(records):
            print(json.dumps(rec, indent=2))
            if i >= 2:  # show only first 3 records
                break


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. We reference all entities by their `@id`, as per Croissant convention.

In [ ]:
# Extract data from each record set

# Gather all RecordSet @ids
record_set_ids = []
if record_sets is not None and len(record_sets) > 0:
    for rs in record_sets:
        record_set_ids.append(rs['@id'])
else:
    print("No RecordSets found.")

dataframes = {}

for rsid in record_set_ids:
    recs = list(dataset.records(record_set=rsid))
    if recs:
        df = pd.DataFrame(recs)
        dataframes[rsid] = df
        print(f"RecordSet {rsid} columns: {df.columns.tolist()}")
        print(df.head())

# Choose the primary record set for EDA (typically the main table in clinical datasets)
primary_record_set_id = record_set_ids[0] if record_set_ids else None


## 4. Exploratory Data Analysis (EDA)
Apply processing steps: filtering, normalization, grouping, and outlier removal.

We select fields for EDA, referencing them by their [`@id`](https://mlcommons.org/croissant/spec/#id).

In [ ]:
# EDA: Use numeric and categorical fields (by @id)

import numpy as np

# Find a numeric field (e.g., age, diagnosis interval, comorbidity count, etc.)
numeric_field_id = None
group_field_id = None
if record_sets and len(record_sets) > 0:
    for f in record_sets[0]['field']:
        if f.get('dataType', '').lower() in ['integer', 'number', 'float']:
            numeric_field_id = f['@id']
            break
    # Choose a group/categorical field (e.g., anatomical location, or MSI status)
    for f in record_sets[0]['field']:
        if f.get('dataType', '').lower() in ['text', 'string']:
            group_field_id = f['@id']
            break
else:
    print('No RecordSets or Fields found.')

df = dataframes[primary_record_set_id] if primary_record_set_id in dataframes else None
if df is not None and numeric_field_id in df.columns:
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field (if present)
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print('Suitable numeric field not found in the DataFrame.')


## 5. Visualization
Visualize data distributions or relationships using matplotlib & seaborn.

In [ ]:
# Visualization: Histogram and boxplot for numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,6))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot grouped by the group field
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We successfully loaded and explored the FAIR^2 dataset using Croissant schema.
Key steps included:
- Loading Croissant metadata
- Listing RecordSets and Fields by their `@id` values
- Extracting main tabular data
- Filtering and normalizing numeric variables
- Grouping and visualizing key attributes

This notebook demonstrates reproducible, FAIR-compliant exploration of clinical datasets. Further analyses can be extended using the rich metadata structure.